# 03 · Model selection & training workbench

Goal: find the **right data treatment** and the **right model** for the symptom → disease classifier, with evidence
you can explain in the demo. Everything here uses the same preprocessing as the app (`app/preprocessing.py`).

**How to use**
1. Run top to bottom (`FAST` mode below gives a ~1-minute smoke run; the default full run takes a few minutes).
2. Read the markdown before each experiment, look at the tables/plots, and fill in the decision tables.
3. Only at the end (section 10) set `CFG["SAVE"] = True` to export the chosen model to `artifacts/`.

**Sections:** 1 data · 2 evaluation protocol · 3 data-handling experiments (leakage, augmentation, balancing,
feature weighting) · 4 model zoo · 5 hyperparameter tuning · 6 calibration · 7 error analysis ·
8 learning curve & robustness · 9 statistical comparison · 10 final model & export · 11 README checklist.

**Environment variables (optional):** `HEALTHAI_FAST=1` quick run · `HEALTHAI_SLOW=1` also try gradient boosting.

In [ ]:
import os, io, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA, ART = ROOT / "data", ROOT / "artifacts"
from app.preprocessing import normalize_token      # identical normalizer to the app

FAST = os.getenv("HEALTHAI_FAST", "0") == "1"
CFG = dict(
    SEED=42,
    N_SPLITS=5,                         # min class size is 5 unique rows, so 5 folds is the maximum
    N_REPEATS=1 if FAST else 2,         # repeated CV -> more stable estimates
    N_AUG=12,                           # random symptom-subset copies per training row
    N_TEST_REPS=2 if FAST else 3,       # random symptom subsets drawn per test row and size k
    INCLUDE_SLOW=os.getenv("HEALTHAI_SLOW", "0") == "1",
    TUNE_RF=not FAST,
    SAVE=False,                         # flip to True only in section 10
)
SEED, NJ = CFG["SEED"], -1
print("FAST mode:", FAST, "|", CFG)

## 1 · Load and clean the data
Same cleaning as `training/train.py`: normalize symptom names, then keep **unique** (disease, symptom-set) rows.
`X` is a binary matrix (1 = symptom present). `X_raw` keeps the 4,920 duplicated rows only for the leakage demo.

In [ ]:
raw = pd.read_csv(DATA / "dataset.csv")
sym_cols = [c for c in raw.columns if c.startswith("Symptom_")]

def row_symptoms(r):
    return tuple(sorted({normalize_token(s) for s in r if isinstance(s, str) and s.strip()}))

records = pd.DataFrame({"disease": raw["Disease"].str.strip(),
                        "symptoms": [row_symptoms(r) for r in raw[sym_cols].values]})
unique = records.drop_duplicates().reset_index(drop=True)
vocab = sorted({s for t in records["symptoms"] for s in t})
V = len(vocab); index = {s: i for i, s in enumerate(vocab)}

def to_matrix(sets):
    M = np.zeros((len(sets), V), dtype=np.float32)
    for i, t in enumerate(sets):
        for s in t:
            M[i, index[s]] = 1
    return M

X, y_str = to_matrix(unique["symptoms"]), unique["disease"].values
X_raw, y_raw_str = to_matrix(records["symptoms"]), records["disease"].values
le = LabelEncoder().fit(y_str)
y, y_raw = le.transform(y_str), le.transform(y_raw_str)
C = len(le.classes_)

print(f"raw rows={len(records)}  unique rows={len(unique)}  diseases={C}  symptoms={V}")
per_class = pd.Series(y_str).value_counts()
print("unique rows per disease: min", per_class.min(), "| median", per_class.median(), "| max", per_class.max())
print("symptoms per row: median", int(np.median(X.sum(1))), "| max", int(X.sum(1).max()))

# severity weights (used only in the feature-weighting experiment)
sev = pd.read_csv(DATA / "Symptom-severity.csv")
sev_map = {normalize_token(a): w for a, w in zip(sev["Symptom"], sev["weight"])}
sev_default = float(np.median(list(sev_map.values())))
SEV_W = np.array([sev_map.get(s, sev_default) for s in vocab], dtype=np.float32)
print("symptoms without a severity weight (filled with median):", [s for s in vocab if s not in sev_map])

## 2 · Evaluation protocol (read this before trusting any number)

* **Unique rows only, split first, augment later.** Augmentation happens *inside* each training fold, never on test rows.
* **Repeated stratified K-fold** (every disease appears in every fold). All models and experiments see the *same* folds and the
  *same* test inputs, so differences are paired and fair.
* **Realistic test inputs.** Users type a few symptoms, not 17. Each test row is turned into random subsets of size
  k = 1…5 (`k1`…`k5`), plus `full` (all symptoms) and `noisy3` (3 true symptoms + 1 wrong one). **Headline = `k3`.**
* **Metrics**
  * `top1`, `top3`, `top5` – is the true disease the first / in the first 3 / 5 suggestions? (`top3` matters most: the app shows a ranked list)
  * `macro_f1` – top-1 quality averaged over diseases (not dominated by easy classes)
  * `logloss`, `brier` – quality of the *probabilities* (the UI shows percentages)
  * `ece`, `conf_gap` – calibration: `conf_gap` = average confidence − accuracy (negative = under-confident)

In [ ]:
RESULTS, PRED, _TRAIN_CACHE = [], {}, {}
rskf = RepeatedStratifiedKFold(n_splits=CFG["N_SPLITS"], n_repeats=CFG["N_REPEATS"], random_state=SEED)
FOLDS = list(rskf.split(X, y))
print("folds:", len(FOLDS))

# ---------- test inputs: identical for every model ----------
def make_subset(M, k, rng, n_wrong=0):
    out = np.zeros_like(M)
    for i, row in enumerate(M):
        ones = np.flatnonzero(row)
        out[i, rng.choice(ones, size=min(k, len(ones)), replace=False)] = 1
        if n_wrong:
            out[i, rng.choice(np.flatnonzero(row == 0), size=n_wrong, replace=False)] = 1
    return out

def build_eval_sets(te_idx, seed):
    rng = np.random.default_rng(seed)
    Xt, yt = X[te_idx], y[te_idx]
    reps = CFG["N_TEST_REPS"]
    sets = {"full": (Xt, yt)}
    for k in (1, 2, 3, 4, 5):
        sets[f"k{k}"] = (np.vstack([make_subset(Xt, k, rng) for _ in range(reps)]), np.tile(yt, reps))
    sets["noisy3"] = (np.vstack([make_subset(Xt, 3, rng, n_wrong=1) for _ in range(reps)]), np.tile(yt, reps))
    return sets

EVAL = [build_eval_sets(te, SEED + i) for i, (tr, te) in enumerate(FOLDS)]

# ---------- training data preparation (inside the fold) ----------
def augment_matrix(Xtr, ytr, n_aug, rng, min_keep=1):
    # each copy keeps a random subset (>= min_keep) of the row's symptoms -> simulates short user input
    ones = [np.flatnonzero(r) for r in Xtr]
    parts_X, parts_y = [Xtr], [ytr]
    for _ in range(n_aug):
        Z = np.zeros_like(Xtr)
        for i, idx in enumerate(ones):
            k = rng.integers(min(min_keep, len(idx)), len(idx) + 1)
            Z[i, rng.choice(idx, size=k, replace=False)] = 1
        parts_X.append(Z); parts_y.append(ytr)
    return np.vstack(parts_X), np.concatenate(parts_y)

def oversample_balance(Xtr, ytr, rng):
    target = np.bincount(ytr, minlength=C).max(); idx = []
    for c in range(C):
        ci = np.flatnonzero(ytr == c)
        if len(ci) == 0:
            continue
        idx.extend(ci)
        if target - len(ci) > 0:
            idx.extend(rng.choice(ci, size=target - len(ci), replace=True))
    idx = np.array(idx)
    return Xtr[idx], ytr[idx]

def subsample(tr, frac, rng):
    keep = []
    for c in np.unique(y[tr]):
        ci = tr[y[tr] == c]
        keep.extend(rng.choice(ci, size=min(len(ci), max(2, int(np.ceil(frac * len(ci))))), replace=False))
    return np.array(keep)

def prepare_train(i, n_aug, balance, frac, min_keep):
    key = (i, n_aug, balance, frac, min_keep)
    if key not in _TRAIN_CACHE:
        rng = np.random.default_rng(SEED + 1000 + i)
        tr = FOLDS[i][0]
        if frac < 1:
            tr = subsample(tr, frac, rng)
        Xo, yo = X[tr], y[tr]
        Xt, yt = (oversample_balance(Xo, yo, rng) if balance else (Xo, yo))
        if n_aug:
            Xt, yt = augment_matrix(Xt, yt, n_aug, rng, min_keep)
        _TRAIN_CACHE[key] = (Xt, yt, Xo)
    return _TRAIN_CACHE[key]

# ---------- metrics ----------
def proba(model, A):
    P = model.predict_proba(A)
    if P.shape[1] != C:                       # a class missing from training -> pad with zeros
        full = np.zeros((len(A), C)); full[:, model.classes_] = P; P = full
    return P

def metrics_from_proba(P, yt):
    n = len(yt); order = np.argsort(-P, axis=1); hit1 = order[:, 0] == yt
    onehot = np.zeros_like(P); onehot[np.arange(n), yt] = 1
    conf = P.max(1); ece = 0.0; bins = np.linspace(0, 1, 11)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(hit1[m].mean() - conf[m].mean())
    return {"top1": hit1.mean(),
            "top3": (order[:, :3] == yt[:, None]).any(1).mean(),
            "top5": (order[:, :5] == yt[:, None]).any(1).mean(),
            "macro_f1": f1_score(yt, order[:, 0], average="macro", zero_division=0),
            "logloss": -np.log(np.clip(P[np.arange(n), yt], 1e-12, 1)).mean(),
            "brier": ((P - onehot) ** 2).sum(1).mean(),
            "ece": ece, "conf_gap": conf.mean() - hit1.mean()}

def cv_eval(make_model, name, experiment="zoo", n_aug=None, balance=False, frac=1.0,
            min_keep=1, tf_factory=None, store=False):
    n_aug = CFG["N_AUG"] if n_aug is None else n_aug
    rows, stash = [], {"y": [], "P": [], "fold": []}
    for i in range(len(FOLDS)):
        Xtr, ytr, Xorig = prepare_train(i, n_aug, balance, frac, min_keep)
        tf = tf_factory(Xorig) if tf_factory else (lambda A: A)
        m = make_model(); t0 = time.perf_counter(); m.fit(tf(Xtr), ytr); fit_s = time.perf_counter() - t0
        for set_name, (Xt, yt) in EVAL[i].items():
            P = proba(m, tf(Xt))
            rows.append({"model": name, "experiment": experiment, "fold": i, "eval_set": set_name,
                         "fit_s": fit_s, **metrics_from_proba(P, yt)})
            if store and set_name == "k3":
                stash["y"].append(yt); stash["P"].append(P); stash["fold"].append(np.full(len(yt), i))
    df = pd.DataFrame(rows); RESULTS.append(df)
    if store:
        PRED[name] = {k: np.concatenate(v) for k, v in stash.items()}
    return df

def summarize(df, eval_set="k3", by="model", sort="top3"):
    d = df[df["eval_set"] == eval_set]
    g = d.groupby(by).agg(top1=("top1", "mean"), top3=("top3", "mean"), top3_std=("top3", "std"),
                          top5=("top5", "mean"), macro_f1=("macro_f1", "mean"), logloss=("logloss", "mean"),
                          brier=("brier", "mean"), ece=("ece", "mean"), conf_gap=("conf_gap", "mean"),
                          fit_s=("fit_s", "mean"))
    return g.sort_values(sort, ascending=False).round(3)

def measure(make_model):
    # single-request latency and on-disk size, as the web app would experience them
    Xtr, ytr, _ = prepare_train(0, CFG["N_AUG"], False, 1.0, 1)
    m = make_model().fit(Xtr, ytr)
    if "n_jobs" in m.get_params():
        m.set_params(n_jobs=1)
    Xt = EVAL[0]["k3"][0][:30]
    t0 = time.perf_counter()
    for r in Xt:
        m.predict_proba(r.reshape(1, -1))
    ms = (time.perf_counter() - t0) / len(Xt) * 1000
    buf = io.BytesIO(); joblib.dump(m, buf, compress=3)
    return round(ms, 2), round(len(buf.getvalue()) / 1024)
print("protocol ready:", len(EVAL[0]), "evaluation sets per fold:", list(EVAL[0]))

## 3 · Data-handling experiments
Reference learner for these experiments: **Logistic Regression (C=10)**. Each experiment changes *one* thing about the data.

### 3a · Leakage: why we de-duplicate before splitting
The raw file has 4,920 rows but only a few hundred unique patterns. A random split of the raw rows puts the same pattern in
train and test, so the score is inflated.

In [ ]:
LR = lambda: LogisticRegression(C=10, max_iter=1000)
rng = np.random.default_rng(SEED)

# (a) naive: random split of the raw, duplicated rows
tr_i, te_i = train_test_split(np.arange(len(y_raw)), test_size=0.2, stratify=y_raw, random_state=SEED)
m = LR().fit(X_raw[tr_i], y_raw[tr_i])
leaky_full = metrics_from_proba(proba(m, X_raw[te_i]), y_raw[te_i])
Xs3 = make_subset(X_raw[te_i], 3, rng)
leaky_k3 = metrics_from_proba(proba(m, Xs3), y_raw[te_i])
dup_share = np.mean([tuple(r) in {tuple(t) for t in X_raw[tr_i]} for r in X_raw[te_i][:300]])

# (b) correct: de-duplicate first, no augmentation yet
proper = cv_eval(LR, "LR dedupe-first, no augmentation", experiment="leakage", n_aug=0)
p = summarize(proper, "full").iloc[0]; p3 = summarize(proper, "k3").iloc[0]

out = pd.DataFrame({
    "naive raw split": [leaky_full["top1"], leaky_k3["top1"], leaky_k3["top3"]],
    "dedupe first":    [p["top1"], p3["top1"], p3["top3"]],
}, index=["top1 (all symptoms)", "top1 (3 symptoms)", "top3 (3 symptoms)"]).round(3)
print(f"share of naive test rows that also appear in the train set: {dup_share:.0%}")
display(out)

### 3b · Augmentation by symptom dropout
Training rows have many symptoms; users type few. We add copies of each *training* row with random symptoms removed.
Question: how many copies help, and does it matter for 1–2 symptom inputs?

In [ ]:
aug_runs = []
for n in ([0, 12] if FAST else [0, 4, 12, 30, 60]):
    aug_runs.append(cv_eval(LR, f"aug x{n}", experiment="augmentation", n_aug=n))
aug_df = pd.concat(aug_runs)
tab = aug_df[aug_df["eval_set"].isin(["full", "k1", "k2", "k3", "k5"])].pivot_table(
    index="model", columns="eval_set", values="top1", aggfunc="mean")[["k1", "k2", "k3", "k5", "full"]].round(3)
display(tab)
fig, ax = plt.subplots(figsize=(6, 3.5))
for col in ["k1", "k3", "full"]:
    ax.plot(tab.index, tab[col], marker="o", label=col)
ax.set_title("Top-1 vs augmentation amount"); ax.set_ylabel("top-1"); ax.legend(); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

### 3c · Class balancing
Classes are already balanced (5–10 unique rows each; 120 raw rows each), so resampling should do little. We check
`class_weight="balanced"` and random oversampling instead of assuming. *Not* tested on purpose: SMOTE – interpolating between
binary symptom vectors creates symptom combinations that never occur.

In [ ]:
bal = pd.concat([
    cv_eval(LR, "none", experiment="balancing"),
    cv_eval(lambda: LogisticRegression(C=10, max_iter=1000, class_weight="balanced"), "class_weight=balanced", experiment="balancing"),
    cv_eval(LR, "random oversampling", experiment="balancing", balance=True),
])
display(summarize(bal, "k3")[["top1", "top3", "top3_std", "macro_f1", "logloss", "ece"]])

### 3d · Feature weighting
Binary presence vs severity-weighted (from `Symptom-severity.csv`) vs IDF-weighted (rare symptoms count more; fitted on the
training fold only).

In [ ]:
def tf_binary(Xo):
    return lambda A: A
def tf_severity(Xo):
    return lambda A: A * SEV_W / SEV_W.mean()
def tf_idf(Xo):
    idf = (np.log((len(Xo) + 1) / (Xo.sum(0) + 1)) + 1).astype(np.float32)
    return lambda A: A * idf
feat = pd.concat([
    cv_eval(LR, "binary", experiment="features", tf_factory=tf_binary),
    cv_eval(LR, "severity-weighted", experiment="features", tf_factory=tf_severity),
    cv_eval(LR, "IDF-weighted", experiment="features", tf_factory=tf_idf),
])
display(summarize(feat, "k3")[["top1", "top3", "top3_std", "macro_f1", "logloss", "ece"]])

### Decisions from section 3 (fill in after reading the tables)

| Question | What the numbers showed | Decision |
|---|---|---|
| De-duplicate before splitting? | | |
| How many augmentation copies? | | |
| Any balancing needed? | | |
| Feature weighting? | | |

Rule of thumb: a difference smaller than the fold-to-fold standard deviation (`top3_std`) is a tie – prefer the simpler option.

## 4 · Model zoo
Same folds, same data treatment (`N_AUG` copies, binary features), many algorithms. Tree and kNN probabilities are
vote fractions, so their `logloss`/`ece` are expected to look poor – that is itself a finding for a UI that shows percentages.

In [ ]:
ZOO = {
    "Dummy (prior)":       lambda: DummyClassifier(strategy="prior"),
    "DecisionTree":        lambda: DecisionTreeClassifier(random_state=SEED),
    "RandomForest":        lambda: RandomForestClassifier(200, min_samples_leaf=2, n_jobs=NJ, random_state=SEED),
    "ExtraTrees":          lambda: ExtraTreesClassifier(200, min_samples_leaf=2, n_jobs=NJ, random_state=SEED),
    "LogisticRegression":  lambda: LogisticRegression(C=10, max_iter=1000),
    "BernoulliNB":         lambda: BernoulliNB(alpha=0.5),
    "kNN (k=7, jaccard)":  lambda: KNeighborsClassifier(7, metric="jaccard", algorithm="brute"),
    "MLP (128)":           lambda: MLPClassifier((128,), max_iter=300, random_state=SEED),
}
if CFG["INCLUDE_SLOW"]:
    ZOO["HistGradientBoosting"] = lambda: HistGradientBoostingClassifier(max_iter=100, random_state=SEED)

zoo_runs = []
for name, mk in ZOO.items():
    t0 = time.time(); zoo_runs.append(cv_eval(mk, name, experiment="zoo", store=True))
    print(f"{name:<22} done in {time.time() - t0:5.1f}s")
zoo_df = pd.concat(zoo_runs)
zoo_tab = summarize(zoo_df, "k3")
ms_size = {n: measure(mk) for n, mk in ZOO.items()}
zoo_tab["ms/predict"] = [ms_size[n][0] for n in zoo_tab.index]
zoo_tab["size KB"] = [ms_size[n][1] for n in zoo_tab.index]
display(zoo_tab)
print("headline = eval_set k3 (user types 3 symptoms). Sorted by top3.")

In [ ]:
show = zoo_tab.drop(index="Dummy (prior)", errors="ignore")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].barh(show.index, show["top3"], xerr=show["top3_std"], color="#0b6bcb"); axes[0].set_title("top-3 accuracy (k=3 symptoms)")
axes[0].set_xlim(max(0, show["top3"].min() - 0.05), 1.0)
axes[1].barh(show.index, np.minimum(show["logloss"], 5), color="#d9822b"); axes[1].set_title("log-loss (lower is better, clipped at 5)")
axes[2].barh(show.index, show["conf_gap"], color="#2b9d6a"); axes[2].axvline(0, color="k"); axes[2].set_title("confidence - accuracy (0 = honest)")
plt.tight_layout(); plt.show()

# how accuracy grows with the number of symptoms the user types
curve = zoo_df[zoo_df["eval_set"].isin(["k1", "k2", "k3", "k4", "k5", "full"])].pivot_table(
    index="eval_set", columns="model", values="top3", aggfunc="mean").loc[["k1", "k2", "k3", "k4", "k5", "full"]]
top_models = list(show.index[:5])
ax = curve[top_models].plot(marker="o", figsize=(7, 4), title="top-3 accuracy vs number of symptoms typed")
ax.set_ylabel("top-3"); plt.tight_layout(); plt.show()

## 5 · Hyperparameter tuning
Sweep the main knob of each promising family using the **same paired folds**. Pick by `top3` first, then `logloss`
(probability quality), then simplicity. Beware over-tuning: with ~300 rows, small gaps are noise.

In [ ]:
tune = []
for c in ([1, 10, 100] if FAST else [0.1, 0.3, 1, 3, 10, 30, 100, 300]):
    tune.append(cv_eval(lambda c=c: LogisticRegression(C=c, max_iter=2000), f"LR C={c}", experiment="tune_lr"))
for a in ([0.1, 1] if FAST else [0.05, 0.1, 0.5, 1, 2]):
    tune.append(cv_eval(lambda a=a: BernoulliNB(alpha=a), f"NB alpha={a}", experiment="tune_nb"))
for k in ([5, 11] if FAST else [3, 5, 7, 11, 15]):
    tune.append(cv_eval(lambda k=k: KNeighborsClassifier(k, metric="jaccard", algorithm="brute"), f"kNN k={k}", experiment="tune_knn"))
if CFG["TUNE_RF"]:
    for n_est in (100, 300):
        for leaf in (1, 2, 4):
            for mf in ("sqrt", 0.3):
                tune.append(cv_eval(lambda n_est=n_est, leaf=leaf, mf=mf: RandomForestClassifier(
                    n_est, min_samples_leaf=leaf, max_features=mf, n_jobs=NJ, random_state=SEED),
                    f"RF n={n_est} leaf={leaf} mf={mf}", experiment="tune_rf"))
tune_df = pd.concat(tune)
tune_tab = summarize(tune_df, "k3", sort="top3")
display(tune_tab[["top1", "top3", "top3_std", "logloss", "ece", "conf_gap"]].head(15))

In [ ]:
fam = tune_df[tune_df["eval_set"] == "k3"].copy()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
lr = summarize(tune_df[tune_df["experiment"] == "tune_lr"], "k3", sort="top3")
lr["C"] = [float(i.split("=")[1]) for i in lr.index]; lr = lr.sort_values("C")
axes[0].semilogx(lr["C"], lr["top3"], marker="o", label="top3"); axes[0].set_xlabel("C (inverse regularization)"); axes[0].set_title("Logistic Regression: accuracy vs C")
ax2 = axes[0].twinx(); ax2.semilogx(lr["C"], lr["logloss"], color="tab:red", marker="s"); ax2.set_ylabel("log-loss", color="tab:red")
nb = summarize(tune_df[tune_df["experiment"] == "tune_nb"], "k3", sort="top3")
nb["alpha"] = [float(i.split("=")[1]) for i in nb.index]; nb = nb.sort_values("alpha")
axes[1].plot(nb["alpha"], nb["top3"], marker="o"); axes[1].set_xlabel("alpha (smoothing)"); axes[1].set_title("Bernoulli NB: accuracy vs alpha")
plt.tight_layout(); plt.show()

best_by_family = {}
for exp in tune_df["experiment"].unique():
    t = summarize(tune_df[tune_df["experiment"] == exp], "k3", sort="top3").sort_values(["top3", "logloss"], ascending=[False, True])
    best_by_family[exp] = t.index[0]
print("best configuration per family:", best_by_family)

## 6 · Calibration – can we trust the percentages?
The UI shows "X% match". A model that is right 85% of the time but reports 55% confidence (or 99%) misleads users.
Reliability diagram: points on the diagonal = honest probabilities. We use **out-of-fold** predictions (each row predicted by a
model that never saw it).

In [ ]:
def reliability(P, yt, bins=10):
    conf, hit = P.max(1), (P.argmax(1) == yt); edges = np.linspace(0, 1, bins + 1); xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5:
            xs.append(conf[m].mean()); ys.append(hit[m].mean()); ns.append(m.sum())
    return xs, ys, ns

cal_models = [m for m in ["RandomForest", "LogisticRegression", "BernoulliNB", "MLP (128)"] if m in PRED]
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect")
for m in cal_models:
    xs, ys, _ = reliability(PRED[m]["P"], PRED[m]["y"]); ax.plot(xs, ys, marker="o", label=m)
ax.set_xlabel("mean predicted confidence"); ax.set_ylabel("actual accuracy"); ax.set_title("Reliability (k=3 symptoms)"); ax.legend()
plt.tight_layout(); plt.show()
display(zoo_tab.loc[[m for m in cal_models], ["top1", "logloss", "brier", "ece", "conf_gap"]])

In [ ]:
# Temperature scaling: a one-number fix for mis-calibrated probabilities. Fit T on out-of-fold predictions of
# even-numbered folds, evaluate on odd-numbered folds (so we never score on data used to fit T).
def apply_T(P, T):
    Q = np.power(np.clip(P, 1e-12, 1), 1.0 / T); return Q / Q.sum(1, keepdims=True)

def ll(P, yt):
    return -np.log(np.clip(P[np.arange(len(yt)), yt], 1e-12, 1)).mean()

rows = []
for m in cal_models:
    d = PRED[m]; fit_m, ev_m = d["fold"] % 2 == 0, d["fold"] % 2 == 1
    res = minimize_scalar(lambda T: ll(apply_T(d["P"][fit_m], T), d["y"][fit_m]), bounds=(0.05, 10), method="bounded")
    T = res.x; before = metrics_from_proba(d["P"][ev_m], d["y"][ev_m]); after = metrics_from_proba(apply_T(d["P"][ev_m], T), d["y"][ev_m])
    rows.append({"model": m, "T": round(T, 2), "logloss before": round(before["logloss"], 3), "logloss after": round(after["logloss"], 3),
                 "ece before": round(before["ece"], 3), "ece after": round(after["ece"], 3)})
display(pd.DataFrame(rows).set_index("model"))
print("T < 1 sharpens an under-confident model; T > 1 softens an over-confident one. top-1/top-3 never change (ranking is unchanged).")

## 7 · Error analysis – *where* does the best model fail?
Pick the model to analyse (default: best `top3` in the zoo, excluding the dummy baseline). Confusions between diseases that
share most symptoms are expected and are why the app returns a ranked list instead of a single answer.

In [ ]:
CHOSEN = None   # e.g. "LogisticRegression"; None -> automatic
BEST = CHOSEN or zoo_tab.drop(index="Dummy (prior)").sort_values(["top3", "logloss"], ascending=[False, True]).index[0]
print("analysing:", BEST)
d = PRED[BEST]; pred = d["P"].argmax(1)

cm = confusion_matrix(d["y"], pred, labels=np.arange(C)); cmn = cm / cm.sum(1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(11, 9)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(C)); ax.set_xticklabels(le.classes_, rotation=90, fontsize=6); ax.set_yticks(range(C)); ax.set_yticklabels(le.classes_, fontsize=6)
ax.set_title(f"{BEST}: normalized confusion matrix (k=3 symptoms)"); ax.set_xlabel("predicted"); ax.set_ylabel("true"); plt.colorbar(im, shrink=0.7)
plt.tight_layout(); plt.show()

conf_pairs = [(le.classes_[i], le.classes_[j], int(cm[i, j])) for i in range(C) for j in range(C) if i != j and cm[i, j] > 0]
conf_df = pd.DataFrame(sorted(conf_pairs, key=lambda t: -t[2])[:12], columns=["true", "predicted as", "count"])
sets = unique["symptoms"].groupby(unique["disease"]).apply(lambda s: set().union(*s))
conf_df["symptom overlap (Jaccard)"] = [round(len(sets[a] & sets[b]) / len(sets[a] | sets[b]), 2) for a, b in zip(conf_df["true"], conf_df["predicted as"])]
display(conf_df)

recall = pd.Series(np.diag(cm) / cm.sum(1).clip(1), index=le.classes_).sort_values()
fig, ax = plt.subplots(figsize=(7, 9)); recall.plot.barh(ax=ax, color="#0b6bcb"); ax.set_title("Per-disease top-1 recall (k=3)")
plt.tight_layout(); plt.show()
print("hardest diseases:", list(recall.index[:5]))

## 8 · Learning curve and robustness
* **Learning curve:** would more data help, or have we saturated? (Rows are *unique* patterns, so this is the real data size.)
* **Robustness:** a user adds one wrong symptom (`noisy3`). Models that over-rely on a single feature drop more.

In [ ]:
fracs = [0.4, 1.0] if FAST else [0.2, 0.4, 0.6, 0.8, 1.0]
lc_models = {k: ZOO[k] for k in ["LogisticRegression", "BernoulliNB", "RandomForest"]}
lc_rows = []
for name, mk in lc_models.items():
    for f in fracs:
        t = summarize(cv_eval(mk, f"{name}@{f}", experiment="learning_curve", frac=f), "k3").iloc[0]
        lc_rows.append({"model": name, "train fraction": f, "top3": t["top3"], "top1": t["top1"]})
lc = pd.DataFrame(lc_rows)
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, g in lc.groupby("model"):
    ax.plot(g["train fraction"], g["top3"], marker="o", label=name)
ax.set_xlabel("fraction of unique training rows"); ax.set_ylabel("top-3 (k=3)"); ax.set_title("Learning curve"); ax.legend()
plt.tight_layout(); plt.show()

rob = zoo_df[zoo_df["eval_set"].isin(["k3", "noisy3"])].pivot_table(index="model", columns="eval_set", values="top3", aggfunc="mean")
rob["drop with 1 wrong symptom"] = (rob["k3"] - rob["noisy3"]).round(3)
display(rob.drop(index="Dummy (prior)", errors="ignore").sort_values("drop with 1 wrong symptom").round(3))

## 9 · Is the winner really better? (paired comparison)
Compare the top candidates fold by fold. Repeated-CV folds are not fully independent, so treat the p-value as a rough guide:
if the mean difference is smaller than about two standard errors, call it a **tie** and choose on the other criteria
(calibration, size, speed, explainability).

In [ ]:
cands = list(zoo_tab.drop(index="Dummy (prior)").sort_values(["top3", "logloss"], ascending=[False, True]).index[:4])
def per_fold(name, metric="top3"):
    d = zoo_df[(zoo_df["model"] == name) & (zoo_df["eval_set"] == "k3")].sort_values("fold")
    return d[metric].values
rows = []
ref = cands[0]
for other in cands[1:]:
    for metric in ("top3", "logloss"):
        a, b = per_fold(ref, metric), per_fold(other, metric); diff = a - b
        t, p = stats.ttest_rel(a, b) if np.any(diff) else (0, 1)
        rows.append({"reference": ref, "vs": other, "metric": metric, "mean diff": round(diff.mean(), 4),
                     "std": round(diff.std(ddof=1), 4), "folds ref better": int((diff > 0).sum() if metric == "top3" else (diff < 0).sum()),
                     "n folds": len(diff), "p-value": round(float(p), 3)})
display(pd.DataFrame(rows))
print("log-loss: lower is better, so a negative mean diff favors the reference.")

## 10 · Final model and export
Fill the decision in `FINAL` below. The notebook then trains on **all unique rows** (plus augmentation), shows the cross-validated
numbers for that configuration, and — only if `CFG["SAVE"] = True` — writes `artifacts/model.joblib`,
`vocabulary.json` and `metrics.json` (existing artifacts are backed up first).

Important details the app depends on:
* the saved object must be a **plain scikit-learn estimator** with `predict_proba` and `classes_` (disease *names*);
* `vocabulary.json` order must match the feature order used for training (it does: `sorted` symptom names);
* the app loads it with `SymptomModel()` → `joblib.load`, so no notebook code is needed at serving time.

In [ ]:
FINAL = {
    "name": "LogisticRegression",                       # <- your decision
    "make": lambda: LogisticRegression(C=10, max_iter=2000),   # <- your tuned hyperparameters
    "n_aug": CFG["N_AUG"],
}

# cross-validated estimate for exactly this configuration
final_cv = cv_eval(FINAL["make"], "FINAL", experiment="final", n_aug=FINAL["n_aug"], store=True)
display(summarize(final_cv, "k3"))
display(summarize(final_cv, "full"))

# fit on everything (string labels so classes_ are disease names, as the app expects)
rng = np.random.default_rng(SEED)
Xa, ya = augment_matrix(X, y, FINAL["n_aug"], rng)
final_model = FINAL["make"]().fit(Xa, le.inverse_transform(ya))
print("classes:", len(final_model.classes_), "| first:", list(final_model.classes_[:3]))

metrics = {"model": FINAL["name"], "params": final_model.get_params(), "n_unique_rows": int(len(X)), "n_augmented_rows": int(len(Xa)),
           "cv": {s: summarize(final_cv, s).iloc[0].to_dict() for s in ["full", "k1", "k3", "k5", "noisy3"]},
           "protocol": f"{CFG['N_SPLITS']}-fold x {CFG['N_REPEATS']} repeated stratified CV on unique rows; augmentation inside folds",
           "note": "dataset is largely synthetic and clean; real-world accuracy will be lower"}
print(json.dumps(metrics["cv"]["k3"], indent=1))

In [ ]:
if CFG["SAVE"]:
    import shutil, datetime
    ART.mkdir(exist_ok=True)
    old = [p for p in ART.glob("*") if p.is_file()]
    if old:
        backup = ART / ("backup_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")); backup.mkdir()
        for p in old:
            shutil.copy2(p, backup / p.name)
        print("backed up previous artifacts to", backup)
    joblib.dump(final_model, ART / "model.joblib", compress=3)
    (ART / "vocabulary.json").write_text(json.dumps(vocab, indent=1))
    (ART / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))
    print("saved:", [p.name for p in ART.glob("*") if p.is_file()], "| model size KB:", round((ART / "model.joblib").stat().st_size / 1024))
else:
    print("CFG['SAVE'] is False -> nothing written. Set it to True and re-run this cell when you are happy with FINAL.")

In [ ]:
# Sanity check: load exactly what the web app will load and try three realistic queries.
# (needs the app's dependencies: pip install -r requirements.txt)
try:
    from app.model import SymptomModel
    sm = SymptomModel()
    for q in ["fever, cough, chills", "itching, skin rash", "chest pain, breathlessness"]:
        r = sm.preprocessor.process(q)
        print(q, "->", [(c.name, c.probability) for c in sm.predict(r.vector, 3)])
except Exception as e:
    print("skipped app check:", type(e).__name__, e)

## 11 · README checklist (copy your answers into the README "Data and model" and "Trade-offs" sections)

| Item | Your answer |
|---|---|
| Final model + hyperparameters | |
| Why this over the runner-up (accuracy / calibration / size / speed / explainability) | |
| Headline CV result (k=3: top-1, top-3, log-loss) | |
| Data decisions (dedupe-first, augmentation copies, balancing, features) and the evidence | |
| Known weaknesses (hardest diseases, overlap-driven confusions) | |
| Limitation to state honestly | Dataset is largely synthetic and clean; real-world accuracy will be lower. |
| Probabilities trustworthy? (conf_gap / ECE; was calibration applied?) | |